# Домашняя работа 4

## Подготовка

In [1]:
from pathlib import Path
import json
import hashlib
from copy import deepcopy
import requests
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = DATA
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.2 (6830, 7)


## Исходники домашней работы

In [2]:
manifest = json.loads((LOCAL / "manifest.json").read_text(encoding="utf-8"))
cache = {}
for info in manifest["pages"]:
    cache[info["request_url"]] = json.loads(
        (LOCAL / info["file"]).read_text(encoding="utf-8")
    )
def cached_loader(url):
    return cache[url]
print(manifest["selected_lei"])
print("Сохранённых страниц:", len(cache))

['549300VZVN841I2ILS84', '549300P1347BFCBHJV60', '549300RPOGWJRH63HS39']
Сохранённых страниц: 2


## Готовая функция обхода страниц

In [3]:
def collect_pages(first_url, loader, max_pages=10):
    url = first_url
    seen_urls = set()
    records = []
    expected_total = None
    for page_number in range(max_pages):
        if url is None:
            break
        if url in seen_urls:
            raise ValueError("Цикл next")
        seen_urls.add(url)
        page = loader(url)
        if not isinstance(page.get("data"), list):
            raise ValueError("data должен быть списком")
        total = page["meta"]["pagination"]["total"]
        if expected_total is None:
            expected_total = total
        if total != expected_total:
            raise ValueError("Число записей изменилось между страницами")
        records.extend(page["data"])
        url = page["links"].get("next")
    if url is not None:
        raise ValueError("Предел страниц: получение не завершено")
    ids = [record["id"] for record in records]
    if len(ids) != len(set(ids)):
        raise ValueError("Повтор ключа между страницами")
    if len(records) != expected_total:
        raise ValueError("Неполный набор")
    return records

## H1 Получение полного справочника

In [4]:
home_records = collect_pages(manifest["first_url"], cached_loader)

home_lookup = (
    pd.json_normalize(home_records)[
        [
            "id",
            "attributes.entity.legalName.name",
            "attributes.entity.legalAddress.country",
        ]
    ]
    .rename(
        columns={
            "id": "lei",
            "attributes.entity.legalName.name": "legal_name_current",
            "attributes.entity.legalAddress.country": "legal_country_current",
        }
    )
)
home_lookup["lei"] = home_lookup["lei"].astype("string")

# Проверки известности, уникальности и точного совпадения с selected_lei
assert home_lookup["lei"].notna().all() and (home_lookup["lei"].str.len() > 0).all()
assert not home_lookup["lei"].duplicated().any()
assert set(home_lookup["lei"]) == set(manifest["selected_lei"])

print("Записей в home_lookup:", len(home_lookup))
display(home_lookup)

Записей в home_lookup: 3


,lei,legal_name_current,legal_country_current
0,549300P1347BFCBHJV60,MASS. INSTITUTE OF TECH. FEDERAL CREDIT UNION,US
1,549300VZVN841I2ILS84,"CROSSCOUNTRY MORTGAGE, LLC",US
2,549300RPOGWJRH63HS39,UNION HOME MORTGAGE CORP.,US


## H2 Пакет исходников и таблица

In [5]:
hw_dir = WORK / "homework"
raw_dir = hw_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Сверка SHA-256 и побайтовое копирование исходных страниц JSON
for info in manifest["pages"]:
    src_path = LOCAL / info["file"]
    raw_bytes = src_path.read_bytes()
    actual_sha256 = hashlib.sha256(raw_bytes).hexdigest()
    assert actual_sha256 == info["sha256"], f"Не совпал SHA-256 для {info['file']}"
    (raw_dir / info["file"]).write_bytes(raw_bytes)

# Копирование исходного manifest.json
(hw_dir / "manifest.json").write_bytes((LOCAL / "manifest.json").read_bytes())

# Сохранение нормализованной таблицы и проверка повторным чтением
lookup_path = hw_dir / "lookup.csv"
home_lookup.to_csv(lookup_path, index=False)

lookup_reloaded = pd.read_csv(lookup_path, dtype={"lei": "string"})
assert_frame_equal(home_lookup, lookup_reloaded)

print("Пакет сохранён в:", hw_dir)
print("SHA-256 страниц и assert_frame_equal успешно подтверждены.")

Пакет сохранён в: ..\results\homework
SHA-256 страниц и assert_frame_equal успешно подтверждены.


## H3 Покрытие записей и организаций

In [6]:
joined = decisions.merge(
    home_lookup,
    on="lei",
    how="left",
    validate="many_to_one",
    indicator=True,
)

# Проверка сохранности исходных полей и числа строк
assert len(joined) == len(decisions)
assert_frame_equal(joined[decisions.columns], decisions)

matched_records = int((joined["_merge"] == "both").sum())
total_records = int(len(decisions))
matched_lei = int(joined.loc[joined["_merge"] == "both", "lei"].nunique())
total_lei = int(decisions["lei"].nunique())

record_coverage = matched_records / total_records
org_coverage = matched_lei / total_lei

coverage_summary = {
    "matched_records": matched_records,
    "total_records": total_records,
    "matched_lei": matched_lei,
    "total_lei": total_lei,
    "record_coverage": record_coverage,
    "org_coverage": org_coverage,
}

(hw_dir / "coverage.json").write_text(
    json.dumps(coverage_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Покрытие по записям: {matched_records} / {total_records} ({record_coverage:.4%})")
print(f"Покрытие по организациям (LEI): {matched_lei} / {total_lei} ({org_coverage:.4%})")

Покрытие по записям: 76 / 6830 (1.1127%)
Покрытие по организациям (LEI): 3 / 314 (0.9554%)


## H+ Проверки испорченных копий страниц